# Тестирование "деревянных" моделей и бустингов.

In [1]:
import polars as pl
import math
from main_funcs import global_temporal_split

In [2]:
from catboost import CatBoostRanker, Pool
import optuna

/home/alex/VS_python_projects/DreamTeamRecSys/.venv313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
events_path = "../../data/dataset/small/marketplace/events"
items_path = "../../data/dataset/small/marketplace/items.pq"

In [4]:
events_df = pl.scan_parquet(events_path)
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String)])

In [5]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os
duration[μs],u64,str,str,str,str
1082d 89901µs,59328774,"""nfmcg_14777696""","""u2i""","""view""","""android"""
1082d 163483µs,66295302,"""nfmcg_26098539""","""catalog""","""view""","""android"""
1082d 864625µs,37542104,"""nfmcg_10840247""","""u2i""","""view""","""android"""
1082d 889192µs,35193548,"""nfmcg_8040572""","""catalog""","""view""","""android"""
1082d 936008µs,27256137,"""nfmcg_6770239""","""catalog""","""view""","""android"""


In [6]:
actions_count = events_df.group_by("action_type").agg(pl.len()).collect(engine="streaming")
actions_count.head()

action_type,len
str,u32
"""view""",126147783
"""clickout""",485448
"""click""",3772094
"""like""",41792


Классы явно не сбалансированны, просмотров гораздо больше. Чтоб сформировать таргет, нужно превратить частоту действия в числовой таргет так, чтобы редкие действия получали больший вес, а частые - меньший. Общее число наблюдений мы делим на число наблюдений этого конкретного действия.

Беру логарифмированный таргет, так как он показал наилучшие результаты в бейзлайне.

In [7]:
actions_count = actions_count.with_columns(
    (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target") 
).drop("len")
actions_count

action_type,log_target
str,f64
"""view""",0.710044
"""clickout""",5.597366
"""click""",3.571844
"""like""",8.046339


In [8]:
events_df = events_df.join(actions_count.lazy(), on="action_type").with_columns([
    (pl.col("action_type") == "view").cast(pl.Int8).alias("target_view"),
    (pl.col("action_type") == "clickout").cast(pl.Int8).alias("target_clickout"),
    (pl.col("action_type") == "like").cast(pl.Int8).alias("target_like"),
    (pl.col("action_type") == "click").cast(pl.Int8).alias("target_click"),
])
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,log_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,f64,i8,i8,i8,i8
1082d 14h 29m 41s 884607µs,6311165,"""nfmcg_8233268""","""u2i""","""view""","""ios""",0.710044,1,0,0,0
1082d 14h 29m 42s 41697µs,70180630,"""nfmcg_4436221""","""other""","""view""","""android""",0.710044,1,0,0,0
1082d 14h 29m 42s 61511µs,50356980,"""nfmcg_16290354""","""u2i""","""view""","""android""",0.710044,1,0,0,0
1082d 14h 29m 42s 259461µs,37954007,"""nfmcg_27598210""","""other""","""view""","""android""",0.710044,1,0,0,0
1082d 14h 29m 42s 607942µs,27939713,"""nfmcg_17780904""","""u2i""","""view""","""android""",0.710044,1,0,0,0


In [9]:
null_info = events_df.null_count().collect()
print(null_info)

shape: (1, 11)
┌───────────┬─────────┬─────────┬───────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ timestamp ┆ user_id ┆ item_id ┆ subdomain ┆ … ┆ target_vie ┆ target_cli ┆ target_lik ┆ target_cl │
│ ---       ┆ ---     ┆ ---     ┆ ---       ┆   ┆ w          ┆ ckout      ┆ e          ┆ ick       │
│ u32       ┆ u32     ┆ u32     ┆ u32       ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│           ┆         ┆         ┆           ┆   ┆ u32        ┆ u32        ┆ u32        ┆ u32       │
╞═══════════╪═════════╪═════════╪═══════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 0         ┆ 0       ┆ 0       ┆ 41792     ┆ … ┆ 0          ┆ 0          ┆ 0          ┆ 0         │
└───────────┴─────────┴─────────┴───────────┴───┴────────────┴────────────┴────────────┴───────────┘


Посмотрю по таблице items, какие есть пропуски.

In [10]:
ivents_lf = pl.scan_parquet(items_path)
ivents_lf.head().collect(engine="streaming")

item_id,brand_id,category,subcategory,price,embedding
str,u64,str,str,f64,"array[f32, 300]"
"""nfmcg_10000001""",137356,null,null,6.06364,"[-0.070853, 0.03246, … 0.082178]"
"""nfmcg_10000012""",53389,null,null,0.960436,"[-0.091099, 0.043168, … 0.060287]"
"""nfmcg_1000002""",34153,"""Fashion Accessories, Tech Add-…","""Hats, Scarves, and Shawls""",-1.793113,"[-0.056533, 0.082878, … 0.037475]"
"""nfmcg_10000034""",39543,null,null,3.416755,"[-0.112588, 0.043333, … -0.001615]"
"""nfmcg_10000039""",82320,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",4.293433,"[-0.157201, 0.026234, … 0.013904]"


In [11]:
null_info = ivents_lf.null_count().collect()
print("Количество строк:", ivents_lf.select(pl.len()).collect().item())
print("Количество колонок:", len(ivents_lf.columns))
print(null_info)

Количество строк: 2325409
Количество колонок: 6
shape: (1, 6)
┌─────────┬──────────┬──────────┬─────────────┬───────┬───────────┐
│ item_id ┆ brand_id ┆ category ┆ subcategory ┆ price ┆ embedding │
│ ---     ┆ ---      ┆ ---      ┆ ---         ┆ ---   ┆ ---       │
│ u32     ┆ u32      ┆ u32      ┆ u32         ┆ u32   ┆ u32       │
╞═════════╪══════════╪══════════╪═════════════╪═══════╪═══════════╡
│ 0       ┆ 0        ┆ 966395   ┆ 1233023     ┆ 2882  ┆ 73        │
└─────────┴──────────┴──────────┴─────────────┴───────┴───────────┘


/tmp/ipykernel_823292/2265000357.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок:", len(ivents_lf.columns))


Подготовлю данные для дальнейшей работы с моделями. Категориальные признаки оставляю, так как бустинги умеют с ними работать. Удаляю товары с пропусками в цене, так как их минимальное количество от общего числа. Эмбеддинги пока не беру, пропуски в категориях и подкатегориях заменяю на новую категорию "unknown". Строки в таблице действий с пропусками в поддомене также просто удаляю, так как их не много. Далее форматирую временную отметку в таблоице событий и собираю все в одну таблицу events_prepared, с которой и продолжу работу.

In [12]:
ACCEPTABLE_ACTIONS = ["view", "click", "clickout", "like"]

# Очистка items
items_lf = (
    ivents_lf
    .drop_nulls(subset=["price"])
    .with_columns([
        pl.col("category").fill_null("unknown").alias("category"),
        pl.col("subcategory").fill_null("unknown").alias("subcategory"),
        pl.col("brand_id").fill_null(-1).alias("brand_id"),
    ])
    .select([
        "item_id",
        "brand_id",
        "category",
        "subcategory",
        "price",
        # embedding пока не использую
    ])
)

# Подготовка events
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000

events_prepared = (
    pl.scan_parquet(events_path)
    .filter(pl.col("action_type").is_in(ACCEPTABLE_ACTIONS))
    .join(actions_count.lazy(), on="action_type", how="inner")
    .join(items_lf, on="item_id", how="inner")
    .with_columns([
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).alias("day"),
    ])
    .filter(pl.col("subdomain").is_not_null())
    .select([
        "timestamp",
        "day",
        "user_id",
        "item_id",
        "subdomain",
        "action_type",
        "os",
        "log_target",
        "brand_id",
        "category",
        "subcategory",
        "price",
    ])
)

In [13]:
events_prepared.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('day', Int64),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('log_target', Float64),
        ('brand_id', Int64),
        ('category', String),
        ('subcategory', String),
        ('price', Float64)])

In [14]:
events_prepared.head().collect(engine="streaming")

timestamp,day,user_id,item_id,subdomain,action_type,os,log_target,brand_id,category,subcategory,price
duration[μs],i64,u64,str,str,str,str,f64,i64,str,str,f64
1082d 14h 55m 17s 882254µs,1082,84487948,"""nfmcg_4573722""","""u2i""","""view""","""android""",0.710044,165269,"""Miscellaneous Goods (Uncategor…","""unknown""",2.010951
1082d 14h 55m 18s 715307µs,1082,19108842,"""nfmcg_1032756""","""u2i""","""view""","""android""",0.710044,173015,"""Cosmetics, Personal Care, and …","""Facial Cosmetics and Care Prod…",-0.617901
1082d 14h 55m 25s 891823µs,1082,19134561,"""nfmcg_22860181""","""u2i""","""view""","""android""",0.710044,173015,"""Cosmetics, Personal Care, and …","""Perfumes and Aromatic Products""",1.284151
1082d 14h 55m 27s 227782µs,1082,62871998,"""nfmcg_19716549""","""u2i""","""view""","""android""",0.710044,137356,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",2.55626
1082d 14h 55m 27s 687306µs,1082,66094829,"""nfmcg_1032756""","""u2i""","""view""","""android""",0.710044,173015,"""Cosmetics, Personal Care, and …","""Facial Cosmetics and Care Prod…",-0.617901


Беру данные только за 5% последних дней - максимум, который позволяет моя оперативная память (32гб).

In [15]:
#границы дней
raw_events = pl.scan_parquet(events_path)
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000

min_day, max_day = (
    raw_events
    .select(
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).min().alias("min_day"),
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).max().alias("max_day"),
    )
    .collect()
    .row(0)
)

days_all = (max_day - min_day) + 1
n_days_subset = int(days_all * 0.05)
if n_days_subset < 1:
    n_days_subset = 1

subset_min_day = max_day - n_days_subset + 1

events_subset = events_prepared.filter(pl.col("day") >= subset_min_day)

In [16]:
train, test = global_temporal_split(events_subset, 0.2)
print("Количество строк train:", train.select(pl.len()).collect().item())
print("Количество колонок train:", len(train.columns))
print("Количество строк test:", test.select(pl.len()).collect().item())
print("Количество колонок test:", len(test.columns))

Количество строк train: 4044137
Количество колонок train: 12


/tmp/ipykernel_823292/3991756773.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок train:", len(train.columns))


Количество строк test: 1072961
Количество колонок test: 12


/tmp/ipykernel_823292/3991756773.py:5: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок test:", len(test.columns))


In [17]:
# агрегируем события до уровня user-item в train и test
train_ui = (
    train
    .group_by(["user_id", "item_id"])
    .agg([
        pl.col("log_target").max().alias("target"),
        pl.col("day").max().alias("last_day"),       
        pl.col("price").mean().alias("price_mean"),
        pl.col("category").first().alias("category"),
        pl.col("subcategory").first().alias("subcategory"),
        pl.col("brand_id").first().alias("brand_id"),
        pl.col("os").first().alias("os"),
        pl.col("subdomain").first().alias("subdomain"),
        pl.col("action_type").first().alias("last_action_type"),
    ])
)

test_ui = (
    test
    .group_by(["user_id", "item_id"])
    .agg([
        pl.col("log_target").max().alias("target"),
        pl.col("day").max().alias("last_day"),
        pl.col("price").mean().alias("price_mean"),
        pl.col("category").first().alias("category"),
        pl.col("subcategory").first().alias("subcategory"),
        pl.col("brand_id").first().alias("brand_id"),
        pl.col("os").first().alias("os"),
        pl.col("subdomain").first().alias("subdomain"),
        pl.col("action_type").first().alias("last_action_type"),
    ])
)

In [18]:
train_ui.head().collect(engine="streaming")

user_id,item_id,target,last_day,price_mean,category,subcategory,brand_id,os,subdomain,last_action_type
u64,str,f64,i64,f64,str,str,i64,str,str,str
13738698,"""nfmcg_22437058""",0.710044,1303,5.054482,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",117885,"""ios""","""u2i""","""view"""
84684261,"""nfmcg_22437058""",0.710044,1299,5.054482,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",117885,"""android""","""other""","""view"""
77513841,"""nfmcg_22437058""",0.710044,1299,5.054482,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",117885,"""android""","""i2i""","""view"""
41288656,"""nfmcg_22437058""",0.710044,1299,5.054482,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",117885,"""android""","""other""","""view"""
70987996,"""nfmcg_15407987""",0.710044,1299,3.447357,"""Miscellaneous Goods (Uncategor…","""unknown""",23188,"""ios""","""u2i""","""view"""


In [19]:
test_ui.head().collect(engine="streaming")

user_id,item_id,target,last_day,price_mean,category,subcategory,brand_id,os,subdomain,last_action_type
u64,str,f64,i64,f64,str,str,i64,str,str,str
86689353,"""nfmcg_6136132""",0.710044,1308,4.896488,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",117885,"""android""","""u2i""","""view"""
18164275,"""nfmcg_27799335""",0.710044,1308,4.575794,"""unknown""","""unknown""",117885,"""android""","""u2i""","""view"""
11236866,"""nfmcg_16565108""",0.710044,1308,-0.968622,"""unknown""","""unknown""",141256,"""android""","""i2i""","""view"""
87260297,"""nfmcg_6136132""",0.710044,1308,4.896488,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",117885,"""android""","""u2i""","""view"""
2831312,"""nfmcg_13121592""",0.710044,1308,-1.42915,"""Children's Products and Childc…","""Diapers and Hygiene Products""",189615,"""other""","""catalog""","""view"""


Заведу еще валидационную выборку.

In [20]:
min_day, max_day = (
    train_ui.select(
        pl.col("last_day").min().alias("min_day"),
        pl.col("last_day").max().alias("max_day"),
    ).collect().row(0)
)

days_all = (max_day - min_day) + 1
val_days = max(int(days_all * 0.1), 1)
val_min_day = max_day - val_days + 1

train_train_lf = train_ui.filter(pl.col("last_day") < val_min_day)
valid_lf = train_ui.filter(pl.col("last_day") >= val_min_day)

Добавлю новые признаки - агрегаты по пользователям, товарам и истории взаимодействия user-item. Эти признаки позже передам для обучения модели.

In [21]:
# user/popularity агрегаты на train_train
user_aggs = (
    train_train_lf
    .group_by("user_id")
    .agg([
        pl.len().alias("user_hist_len"),
        pl.col("target").mean().alias("user_mean_target"),
    ])
)

item_aggs = (
    train_train_lf
    .group_by("item_id")
    .agg([
        pl.len().alias("item_popularity"),
        pl.col("price_mean").mean().alias("price_mean_item"),
    ])
)

In [22]:
print(user_aggs.head().collect(engine="streaming"))
print("________________________")
print(item_aggs.head().collect(engine="streaming"))

shape: (5, 3)
┌──────────┬───────────────┬──────────────────┐
│ user_id  ┆ user_hist_len ┆ user_mean_target │
│ ---      ┆ ---           ┆ ---              │
│ u64      ┆ u32           ┆ f64              │
╞══════════╪═══════════════╪══════════════════╡
│ 37443283 ┆ 9             ┆ 0.710044         │
│ 75862903 ┆ 6             ┆ 0.710044         │
│ 80547289 ┆ 20            ┆ 0.710044         │
│ 79845522 ┆ 32            ┆ 0.888907         │
│ 67217661 ┆ 136           ┆ 0.794215         │
└──────────┴───────────────┴──────────────────┘
________________________
shape: (5, 3)
┌────────────────┬─────────────────┬─────────────────┐
│ item_id        ┆ item_popularity ┆ price_mean_item │
│ ---            ┆ ---             ┆ ---             │
│ str            ┆ u32             ┆ f64             │
╞════════════════╪═════════════════╪═════════════════╡
│ nfmcg_16600433 ┆ 408             ┆ -1.199708       │
│ nfmcg_9964705  ┆ 23              ┆ -0.838137       │
│ nfmcg_12614839 ┆ 428            

In [23]:
def prep_rank_lf(base_lf: pl.LazyFrame) -> pl.LazyFrame:
    return (
        base_lf
        .join(user_aggs, on="user_id", how="left")
        .join(item_aggs, on="item_id", how="left")
    )

train_rank_lf = prep_rank_lf(train_train_lf)
valid_rank_lf = prep_rank_lf(valid_lf)

train_df = train_rank_lf.collect().to_pandas(use_pyarrow_extension_array=False)
valid_df = valid_rank_lf.collect().to_pandas(use_pyarrow_extension_array=False)

In [24]:
feature_cols = [
    "item_id",
    "price_mean",
    "price_mean_item",
    "category",
    "subcategory",
    "brand_id",
    "user_hist_len",
    "user_mean_target",
    "item_popularity",
    "os",
    "subdomain",
    "last_day",
]

cat_features = ["item_id", "category", "subcategory", "brand_id", "os", "subdomain"]

In [25]:
train_df = train_df.sort_values(["user_id", "item_id"])
valid_df = valid_df.sort_values(["user_id", "item_id"])

Обучаю первую модель- вариант CatBoost для задач ранжирования CatBoostRanker с дефолтной функцией потерь для ранжирования YetiRank, беру ее так как она хорошо работает с категориальными признаками и не требует тонких настроек гиперпараметров, можно сразу увидеть результат для сравнения с нашим бейзлайном - SVD. 

Беру только 5000 самых популярных товаров для кандидатогенерации, тк не хватает оперативной памяти.

In [26]:
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df["target"],
    group_id=train_df["user_id"],
    cat_features=cat_features,
)

valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df["target"],
    group_id=valid_df["user_id"],
    cat_features=cat_features,
)

model = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=15",
    iterations=300,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=50,
    early_stopping_rounds=50,
    task_type="CPU",
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.9765882	best: 0.9765882 (0)	total: 1.24s	remaining: 6m 11s
50:	test: 0.9808377	best: 0.9808377 (50)	total: 52.5s	remaining: 4m 16s
100:	test: 0.9811553	best: 0.9811765 (98)	total: 1m 42s	remaining: 3m 21s
150:	test: 0.9811501	best: 0.9812120 (116)	total: 2m 32s	remaining: 2m 30s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9812119758
bestIteration = 116

Shrink model to first 117 iterations.


CatBoostRanker(depth=8, early_stopping_rounds=50, eval_metric='NDCG:top=15', iterations=300, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=50)

In [27]:
TOP_M_ITEMS = 5000
TOP_K = 15

top_items_lf = (
    train_ui
    .group_by("item_id")
    .agg(pl.len().alias("cnt"))
    .sort("cnt", descending=True)
    .limit(TOP_M_ITEMS)
    .select("item_id")
)

user_test_truth = (
    test_ui
    .group_by("user_id")
    .agg([
        pl.col("item_id").alias("true_items"),
        pl.col("target").alias("relevancy"),
    ])
)

Реализую функцию вычисления по батчам с возможностью выбрать размер батча, число юзеров и top k товаров.

In [28]:
def get_topk_from_toppop(
    model,
    feature_cols,
    cat_features,
    user_test_truth: pl.LazyFrame,
    top_items_lf: pl.LazyFrame,
    batch_size: int = 100, 
    top_k: int = 15,
) -> pl.DataFrame:

    item_features = (
        top_items_lf
        .join(item_aggs, on="item_id", how="left")
        .join(train_ui.select("item_id", "price_mean", "category", "subcategory", "brand_id").unique(subset=["item_id"]), on="item_id", how="left")
    ).collect()

    user_features = (
        user_test_truth.select("user_id").unique()
        .join(user_aggs, on="user_id", how="left")
        .join(train_ui.select("user_id", "os", "subdomain", "last_day").unique(subset=["user_id"]), on="user_id", how="left")
    ).collect()

    user_ids = user_features["user_id"].to_list()
    n_users = len(user_ids)
    n_batches = math.ceil(n_users / batch_size)
    all_rows = []

    for b in range(n_batches):
        s = b * batch_size
        e = min((b + 1) * batch_size, n_users)
        batch_users = user_ids[s:e]
        
        # Берем фичи только нужных юзеров
        batch_user_features = user_features.filter(pl.col("user_id").is_in(batch_users))
        cand_df = batch_user_features.join(item_features, how="cross").to_pandas()
        
        # Заполняем пропуски
        for col in cat_features:
            cand_df[col] = cand_df[col].astype("string").fillna("__nan__")
            
        cand_df["user_hist_len"] = cand_df["user_hist_len"].fillna(0).astype("int32")
        cand_df["item_popularity"] = cand_df["item_popularity"].fillna(0).astype("int32")
        cand_df["user_mean_target"] = cand_df["user_mean_target"].fillna(0.0)
        cand_df["price_mean_item"] = cand_df.get("price_mean_item", cand_df["price_mean"])
        
        # Делаем предикт
        pool = Pool(
            data=cand_df[feature_cols],
            cat_features=cat_features,
        )
        
        cand_df["pred_score"] = model.predict(pool)
        
        # Сортируем и берем top_k
        batch_pl = pl.from_pandas(cand_df[["user_id", "item_id", "pred_score"]])
        batch_topk = (
            batch_pl
            .sort(["user_id", "pred_score"], descending=[False, True])
            .group_by("user_id")
            .agg([
                pl.col("item_id").head(top_k).alias("predicted_items"),
                pl.col("pred_score").head(top_k).alias("predicted_scores"),
            ])
        )
        
        truth_small = user_test_truth.filter(pl.col("user_id").is_in(batch_users)).collect()
        batch_eval = truth_small.join(batch_topk, on="user_id", how="inner")
        all_rows.append(batch_eval)
        
    return pl.concat(all_rows)

In [29]:
def ndcg_at_k15(
    user_based_df: pl.DataFrame,
    relevancy_col: str,
    true_items_col: str,
    predicted_items_col: str,
    predicted_score_col: str,
    top_k: int = 15,
):
    """
    Computes user-based NDCG@k for graded relevance in a recommendation setting.

    Parameters
    ----------
    user_based_df : pl.DataFrame
        Dataframe with user data. Each row must contain user and its lists with: truth
        ground items, their relevancy estimation and model prediction score.
    relevancy_col : str
        Column name contains list of relevancy estimations ground
        truth items (pl.List[float]) for user. Elements order must match `true_items_col`.
    true_items_col : str
        Column name of ground truth items with which user had interactions (pl.List[str]). Relevancy
        of these items must be set in `relevancy_col` respectively. 
    predicted_items_col : str
        Columns name with predicted items (pl.List[str]). Must be set in order matches
        `predicted_score_col`.
    predicted_score_col : str
        Columns name with predicted scores for items in `predicted_items_col` (pl.List[float]).
        Used to sort predictions in descending order.
    top_k : int, optional
        Top k elements to calculate `@k` metric.

    Returns
    -------
    pl.DataFrame
        Columns:
        - ``user_id`` : user identifier;
        - ``ndcg`` : NDCG@k for current user.

    Notes
    -----
    For each user, the function:
    1. Aggregates relevancies for ground-truth items by taking the maximum value for each item.
    2. Joins predicted items with their ground-truth relevancies.
    3. Computes DCG@k using the order induced by the model (sorting by score).
    4. Computes IDCG@k using the ideal order (sorting by ground-truth relevancy).
    5. Returns NDCG@k = DCG@k / IDCG@k, or 0.0 if IDCG@k = 0.
    """
    user_ids = []
    ndcgs = []
    for row in user_based_df.iter_rows(named=True):
        true_items = pl.DataFrame(
            {"truth_items": row[true_items_col], "relevancy": row[relevancy_col]}
        )
        true_items = true_items.group_by("truth_items").agg(
            pl.col("relevancy").max()
        )  # Берем максимальную релевантность для товара
        predictions = (
            pl.DataFrame(
                {"predicted_items": row[predicted_items_col], "score": row[predicted_score_col]}
            )
            .join(
                true_items,
                left_on="predicted_items",
                right_on="truth_items",
                coalesce=True,
                how="left",
            )
            .fill_null(0)
        )
        idcg = (
            predictions.select("relevancy")
            .sort("relevancy", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        dcg = (
            predictions.select("score", "relevancy")
            .sort("score", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        user_ids.append(row["user_id"])
        ndcgs.append(0.0 if idcg == 0 else dcg / idcg)
    return pl.DataFrame({"user_id": user_ids, "ndcg": ndcgs})

In [30]:
def calculate_metrics_new(df, k):
    """
    Расчет Precision@k и Recall@k
    df: датафрейм с колонками predicted_items и true_items
    k: количество рекомендаций (TOPK)
    """
    # Обрезаем список предсказаний до k элементов
    top_k_preds = pl.col("predicted_items").list.head(k)
    
    # Ищем пересечение обрезанного списка с правдой
    hits_expr = top_k_preds.list.set_intersection(pl.col("true_items")).list.len()
    
    # Вычисляем метрики одной командой select
    metrics = df.select(
        # Precision = (кол-во попаданий / k), берем среднее по всем юзерам
        (hits_expr / k).mean().alias('precision'),
        
        # Recall = (кол-во попаданий / длину реального списка), берем среднее
        (hits_expr / pl.col('true_items').list.len()).mean().alias('recall')
    )
    
    precision_val = metrics['precision'].item()
    recall_val = metrics['recall'].item()

    return precision_val, recall_val

In [31]:
topk_df = get_topk_from_toppop(
    model=model,
    feature_cols=feature_cols,
    cat_features=cat_features,
    user_test_truth=user_test_truth,
    top_items_lf=top_items_lf,
    top_k=TOP_K,
)

In [32]:
ndcg_df = ndcg_at_k15(
    user_based_df=topk_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=TOP_K,
)

mean_ndcg = ndcg_df["ndcg"].mean()

precision_k, recall_k = calculate_metrics_new(
    df=topk_df.select(["true_items", "predicted_items"]),
    k=TOP_K,
)

print(f"Test mean NDCG@{TOP_K}:", mean_ndcg)
print(f"Test Precision@{TOP_K}:", precision_k)
print(f"Test Recall@{TOP_K}:", recall_k)

Test mean NDCG@15: 0.19989635714885406
Test Precision@15: 0.06664213766309808
Test Recall@15: 0.21056933977863715


Результаты уже получились очень оптимистичными.

Далее дополнительно подберу оптимальные гиперпараметры с помощью Optuna. Буду подбирать самые важные гиперпараметры: learning rate, глубину деревьев, коэффициент L2 регуляризации.

In [33]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "loss_function": "YetiRank",
        "eval_metric": f"NDCG:top={TOP_K}",
        "iterations": 300,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "random_seed": 42,
        "verbose": 0,
        "task_type": "CPU",
        "early_stopping_rounds": 30,
    }

    model = CatBoostRanker(**params)
    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=False,
    )

    valid_scored = valid_df.copy()
    valid_scored["pred"] = model.predict(valid_df[feature_cols])
    valid_scored = valid_scored.sort_values(["user_id", "pred"], ascending=[True, False])

    pred_df = (
        pl.from_pandas(valid_scored[["user_id", "item_id", "pred"]])
        .group_by("user_id")
        .agg(
            pl.col("item_id").head(TOP_K).alias("predicted_items"),
            pl.col("pred").head(TOP_K).alias("predicted_scores"),
        )
    )

    truth_df = (
        pl.from_pandas(valid_df[["user_id", "item_id", "target"]])
        .group_by("user_id")
        .agg(
            pl.col("item_id").alias("true_items"),
            pl.col("target").alias("relevancy"),
        )
    )

    eval_df = truth_df.join(pred_df, on="user_id", how="inner")

    ndcg_df = ndcg_at_k15(
        user_based_df=eval_df,
        relevancy_col="relevancy",
        true_items_col="true_items",
        predicted_items_col="predicted_items",
        predicted_score_col="predicted_scores",
        top_k=TOP_K,
    )

    return float(ndcg_df["ndcg"].mean())

In [34]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=25,
    timeout=600,
)

In [35]:
print("Best value:", study.best_value)
print("Best params:", study.best_params)
print("Best trial:", study.best_trial.number)

Best value: 0.9884941446299346
Best params: {'learning_rate': 0.023688639503640783, 'depth': 10, 'l2_leaf_reg': 7.587945476302646}
Best trial: 0


Теперь снова обучу модель уже с лучшими подобранными гиперпараметрами.

In [36]:
model_best = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=15",
    iterations=300,
    learning_rate=0.02369,
    depth=10,
    l2_leaf_reg=7.588,
    random_seed=42,
    verbose=50,
    early_stopping_rounds=50,
    task_type="CPU",
)

model_best.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        early_stopping_rounds=50,
        verbose=50,
    )

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.9772831	best: 0.9772831 (0)	total: 1.34s	remaining: 6m 42s
50:	test: 0.9803884	best: 0.9803884 (50)	total: 59.7s	remaining: 4m 51s
100:	test: 0.9806508	best: 0.9806632 (99)	total: 1m 57s	remaining: 3m 51s
150:	test: 0.9809695	best: 0.9809746 (134)	total: 2m 51s	remaining: 2m 49s
200:	test: 0.9811330	best: 0.9811330 (200)	total: 3m 47s	remaining: 1m 52s
250:	test: 0.9811952	best: 0.9812029 (248)	total: 4m 44s	remaining: 55.5s
299:	test: 0.9812620	best: 0.9812630 (297)	total: 5m 39s	remaining: 0us

bestTest = 0.9812629811
bestIteration = 297

Shrink model to first 298 iterations.


CatBoostRanker(depth=10, early_stopping_rounds=50, eval_metric='NDCG:top=15', iterations=300, l2_leaf_reg=7.588, learning_rate=0.02369, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=50)

Снова дважды кончалась память, поэтому решил изменить размер батчей. Считаю тест только на 1000 пользователях.

In [37]:
best_topk_df = get_topk_from_toppop(
    model=model_best,
    feature_cols=feature_cols,
    cat_features=cat_features,
    user_test_truth=user_test_truth,
    top_items_lf=top_items_lf,
    top_k=TOP_K,
)

In [38]:
ndcg_df_best = ndcg_at_k15(
    user_based_df=best_topk_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=TOP_K,
)

mean_best_ndcg = ndcg_df_best["ndcg"].mean()

best_precision_k, best_recall_k = calculate_metrics_new(
    df=best_topk_df.select(["true_items", "predicted_items"]),
    k=TOP_K,
)

print(f"Best mean NDCG@{TOP_K}:", mean_best_ndcg)
print(f"Best Precision@{TOP_K}:", best_precision_k)
print(f"Best Recall@{TOP_K}:", best_recall_k)

Best mean NDCG@15: 0.23089235729658722
Best Precision@15: 0.06738354657741275
Best Recall@15: 0.21396265901437142


Итоговый результат после преминения Optuna на тесте из 5000 популярных товаров по всемм пользователям оказался значительно лучше бейзлайна. Подбор гиперпараметров сработал корректно.

# Лучшие метрики:
NDCG@15: 0.23

Precision@15: 0.067

Recall@15: 0.21